# Test & Robustness Evaluation

Evaluates the saved models **once** on the untouched **test** split, then stress-tests them (noise, perturbations, short text).

Needs: notebooks 1 and 2. **Next:** `4_Detection_and_Visualization.ipynb`

In [ ]:
OUT_DIR = "outputs"
SEED    = 42

In [ ]:
import os, re, glob, time, json, warnings, random
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import ComplementNB
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, roc_curve, confusion_matrix, ConfusionMatrixDisplay)
import joblib
try:
    from IPython.display import display
except ImportError:
    display = print
warnings.filterwarnings("ignore")
random.seed(SEED); np.random.seed(SEED)

OUT = Path(OUT_DIR)
for sub in ["figures", "tables", "models", "data", "generated"]: (OUT / sub).mkdir(parents=True, exist_ok=True)
sns.set_theme(style="whitegrid", context="paper", font_scale=1.25, palette="colorblind")
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 300, "savefig.bbox": "tight"})

def savefig(name):
    plt.savefig(OUT / "figures" / f"{name}.png"); plt.savefig(OUT / "figures" / f"{name}.pdf"); plt.show()

def save_table(df, name, index=True):
    df.to_csv(OUT / "tables" / f"{name}.csv", index=index)
    try: (OUT / "tables" / f"{name}.tex").write_text(df.to_latex(index=index, float_format="%.4f"))
    except Exception: pass
    return df
print("Setup OK")

In [ ]:
# ---- shared helpers (fake = positive class; label 1 = real, 0 = fake) ----
def trunc(s, n): return " ".join(str(s).split()[:n])

def load_splits():
    f = OUT / "data" / "splits.pkl"
    if not f.exists(): raise FileNotFoundError("Run notebook 1 (Data Preparation) first.")
    return joblib.load(f)

_pat = json.loads((OUT / "data" / "scrub_patterns.json").read_text()) if (OUT / "data" / "scrub_patterns.json").exists() else []
LEAK_RE = re.compile("|".join(_pat), flags=re.I) if _pat else None
def scrub(s): return re.sub(r"\s+", " ", LEAK_RE.sub(" ", s)).strip() if LEAK_RE else s

def fake_scores(model, X):
    """higher = more likely FAKE."""
    if hasattr(model, "predict_proba"): return model.predict_proba(X)[:, 0]
    return -model.decision_function(X)

def fake_pred(model, X):
    return (fake_scores(model, X) > (0.5 if hasattr(model, "predict_proba") else 0)).astype(int)

def metrics(y, model, X):
    yf = (y == 0).astype(int); s = fake_scores(model, X); p = fake_pred(model, X)
    return dict(Accuracy=accuracy_score(yf, p), Precision=precision_score(yf, p), Recall=recall_score(yf, p),
                F1=f1_score(yf, p), AUC=roc_auc_score(yf, s))

def stylometric(df, title_col="title", text_col="text"):
    t, x = df[text_col].astype(str), df[title_col].astype(str)
    nw = t.str.split().str.len().clip(lower=1); nc = t.str.len().clip(lower=1)
    f = pd.DataFrame(index=df.index)
    f["n_words"] = nw; f["n_chars"] = nc; f["avg_word_len"] = nc / nw
    f["n_sent"] = t.str.count(r"[.!?]+").clip(lower=1); f["avg_sent_len"] = nw / f["n_sent"]
    f["upper_ratio"] = t.str.count(r"[A-Z]") / nc; f["caps_word_ratio"] = t.str.count(r"\b[A-Z]{3,}\b") / nw
    f["exclam"] = t.str.count("!") / nw; f["question"] = t.str.count(r"\?") / nw
    f["quote"] = t.str.count(r"[\"“”']") / nw; f["digit_ratio"] = t.str.count(r"\d") / nc
    f["punct_ratio"] = t.str.count(r"[^\w\s]") / nc
    f["ttr"] = t.map(lambda s: len(set(s.lower().split())) / max(1, len(s.split())))
    f["title_words"] = x.str.split().str.len(); f["title_upper"] = x.str.count(r"[A-Z]") / x.str.len().clip(lower=1)
    f["title_excl"] = x.str.count("!")
    return f.astype(np.float32)

In [ ]:
splits = load_splits(); test = splits["test"]; yte = test.label.values
B = joblib.load(OUT / "models" / "phase2_models.joblib"); tfidf, mdl, kinds = B["tfidf"], B["models"], B["kinds"]
Xte = tfidf.transform(test.doc); Ste = stylometric(test)
XT = {"tfidf": Xte, "style": Ste}
print(f"test={len(test):,} articles, models: {list(mdl)}")

### 3.1 Final test-set results (Table for the paper)

In [ ]:
rows = [{"Model": n, **metrics(yte, m, XT[kinds[n]])} for n, m in mdl.items()]
te = pd.DataFrame(rows).set_index("Model"); save_table(te.round(4), "t3_test_results"); display(te.round(4))

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(18, 5))
te.plot(kind="bar", ax=ax[0], rot=25, width=.8)
ax[0].set_ylim(max(0.5, float(te.values.min()) - 0.03), 1.005); ax[0].set_title("(a) Test-set performance (fake = positive)")
ax[0].legend(ncol=5, fontsize=7, loc="upper center", bbox_to_anchor=(0.5, -0.4))
for n, m in mdl.items():
    fpr, tpr, _ = roc_curve((yte == 0).astype(int), fake_scores(m, XT[kinds[n]])); ax[1].plot(fpr, tpr, label=f"{n} ({te.loc[n, 'AUC']:.3f})")
ax[1].plot([0, 1], [0, 1], "k--", lw=.8); ax[1].set_xlabel("False positive rate"); ax[1].set_ylabel("True positive rate"); ax[1].set_title("(b) ROC curves"); ax[1].legend(fontsize=8)
best = te["F1"].idxmax()
ConfusionMatrixDisplay(confusion_matrix((yte == 0).astype(int), fake_pred(mdl[best], XT[kinds[best]])), display_labels=["Real", "Fake"]).plot(ax=ax[2], cmap="Blues", colorbar=False)
ax[2].set_title(f"(c) Confusion matrix — {best}"); ax[2].grid(False)
plt.tight_layout(); savefig("fig2_test_performance"); print("Best model on test:", best)

### 3.2 Explainability — what do the detectors look at?

In [ ]:
lr = mdl["LogReg (TF-IDF)"]; fn = np.array(tfidf.get_feature_names_out()); co = lr.coef_[0]
tr_, tf_ = np.argsort(co)[-20:], np.argsort(co)[:20]
plt.figure(figsize=(9, 7)); plt.barh(np.r_[fn[tf_], fn[tr_]], np.r_[-co[tf_], -co[tr_]], color=["#d95f02"] * 20 + ["#1b9e77"] * 20)
plt.title("Most influential n-grams (LogReg, after leakage scrubbing)"); plt.xlabel("← Real      Fake →  (coefficient)"); plt.tight_layout(); savefig("fig3_top_features")

from sklearn.inspection import permutation_importance
pi = permutation_importance(mdl["HistGB (Stylometric)"], Ste.iloc[:3000], yte[:3000], n_repeats=3, random_state=SEED, n_jobs=1)
pis = pd.Series(pi.importances_mean, index=Ste.columns).sort_values()
plt.figure(figsize=(7, 5)); pis.plot(kind="barh", color="#7570b3"); plt.title("Stylometric feature importance (permutation)"); plt.tight_layout(); savefig("fig4_style_importance")

### 3.3 Perturbation robustness

How much does performance fall when the text is noisy or changed?

In [ ]:
rngp = np.random.RandomState(SEED)
tsub = test.sample(min(3000, len(test)), random_state=SEED).reset_index(drop=True); yf_sub = (tsub.label.values == 0).astype(int)
def p_lower_nopunct(s): return re.sub(r"\s+", " ", re.sub(r"[^\w\s]", " ", s.lower()))
def p_dropout(s, p=0.2): w = s.split(); k = [x for x in w if rngp.rand() > p]; return " ".join(k) if k else s
def p_typos(s, p=0.05):
    w = s.split()
    for i, x in enumerate(w):
        if len(x) > 3 and rngp.rand() < p: j = rngp.randint(1, len(x) - 2); w[i] = x[:j] + x[j + 1] + x[j] + x[j + 2:]
    return " ".join(w)
def p_shuffle_sent(s): z = re.split(r"(?<=[.!?])\s+", s); rngp.shuffle(z); return " ".join(z)
variants = {"Original": tsub.doc, "Lowercase, no punctuation": tsub.doc.map(p_lower_nopunct), "20% word dropout": tsub.doc.map(p_dropout),
            "5% typos": tsub.doc.map(p_typos), "Shuffled sentences": tsub.doc.map(p_shuffle_sent),
            "Title removed": tsub.text.map(scrub), "First 50 words only": tsub.doc.map(lambda s: trunc(s, 50))}
tf_models = [n for n in mdl if kinds[n] == "tfidf"]; rows = []
for vname, texts in variants.items():
    Xv = tfidf.transform(texts)
    for n in tf_models:
        p = fake_pred(mdl[n], Xv); rows.append({"Perturbation": vname, "Model": n, "F1": f1_score(yf_sub, p), "Accuracy": accuracy_score(yf_sub, p)})
pert = pd.DataFrame(rows); save_table(pert.round(4), "t6_perturbation_results", index=False)
pv = pert.pivot(index="Perturbation", columns="Model", values="F1").reindex(list(variants))
plt.figure(figsize=(9, 4.6)); sns.heatmap(pv, annot=True, fmt=".3f", cmap="RdYlGn", vmin=max(0.5, pv.values.min() - .02), vmax=1)
plt.title("Robustness to text perturbations (F1, fake = positive)"); plt.tight_layout(); savefig("fig7_perturbation_heatmap"); display(pv.round(4))

### 3.4 Length robustness

Detector quality vs. number of words available (LLM text is often short).

In [ ]:
rows = []
for L in [10, 25, 50, 100, 200, 400, None]:
    texts = tsub.doc if L is None else tsub.doc.map(lambda s: trunc(s, L)); Xv = tfidf.transform(texts)
    for n in tf_models:
        rows.append({"Words": "full" if L is None else L, "x": 800 if L is None else L, "Model": n,
                     "F1": f1_score(yf_sub, fake_pred(mdl[n], Xv)), "AUC": roc_auc_score(yf_sub, fake_scores(mdl[n], Xv))})
ln = pd.DataFrame(rows); save_table(ln.drop(columns="x").round(4), "t7_length_robustness", index=False)
plt.figure(figsize=(7.5, 4.5)); sns.lineplot(data=ln, x="x", y="F1", hue="Model", marker="o"); plt.xscale("log")
plt.xticks(sorted(ln.x.unique()), [str(w) for w in ln.drop_duplicates("x").sort_values("x").Words]); plt.xlabel("Words available to the detector")
plt.title("Detector F1 vs. text length"); plt.tight_layout(); savefig("fig8_length_robustness")
print("✅ Notebook 3 finished. Now open 4_Detection_and_Visualization.ipynb")